# Demo: nhận dạng chữ trong ảnh bằng CRNN + CTC (EN và tiếng Việt có dấu)

Notebook này nạp checkpoint đã huấn luyện (`checkpoints/mid_vi_best.pt`, ~0,54 triệu tham số) và đọc chữ trong ảnh mẫu hoặc ảnh bạn tải lên.

**Lưu ý trung thực:** mô hình chỉ được huấn luyện trên ảnh **sinh bằng TRDG** (chữ tối trên nền sáng, một dòng) trên CPU. Trên tập test sinh nó đạt CER ≈ 7,5% nhưng
trên ảnh thật (chụp, nghiêng, nhiều dòng, nền phức tạp) kết quả có thể kém hơn nhiều. Đây là demo của pipeline, không phải OCR sẵn sàng sản xuất.

**Cách dùng trên Colab:** chạy ô 1 và tải lên `ocr_crnn_stage6.zip` khi được hỏi. Chạy cục bộ: mở notebook từ thư mục `notebooks/` của dự án.

In [ ]:
# 1. Chuẩn bị môi trường: trên Colab, tải lên file zip của dự án; cục bộ thì tự tìm thư mục gốc
import os, sys, zipfile
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('ocr_crnn'):
    from google.colab import files
    up = files.upload()                      # chọn ocr_crnn_stage6.zip
    zipfile.ZipFile(next(iter(up))).extractall('.')
if os.path.exists('ocr_crnn'):
    os.chdir('ocr_crnn')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('Thư mục làm việc:', os.getcwd())
assert os.path.exists('checkpoints/mid_vi_best.pt'), 'Không thấy checkpoints/mid_vi_best.pt: kiểm tra lại thư mục dự án'

In [ ]:
# 2. Cài thư viện (trên Colab; torch thường đã có sẵn)
!pip -q install -r requirements.txt

In [ ]:
# 3. Nạp mô hình
import torch
from src.eval import load_checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, charset, mcfg, ck = load_checkpoint('checkpoints/mid_vi_best.pt', device)
n_par = sum(p.numel() for p in model.parameters())
print(f'Thiết bị {device} | {n_par:,} tham số | {charset.num_classes} lớp đầu ra | ảnh cao {mcfg["img_height"]} px | epoch {ck["epoch"]}')

In [ ]:
# 4. Chạy trên 12 ảnh mẫu (lấy từ tập test sinh; nhãn thật trong samples/labels.tsv) với greedy và beam 10
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from demo import predict
from src.metrics import corpus_metrics

rows = [l.split('\t') for l in open('samples/labels.tsv', encoding='utf-8').read().split('\n') if l]
refs, hy_g, hy_b = [], [], []
fig, axes = plt.subplots(len(rows), 1, figsize=(9, 1.0 * len(rows)))
plt.rcParams['font.family'] = 'DejaVu Sans'
for ax, (fn, ref, _) in zip(axes, rows):
    path = f'samples/{fn}'
    g = predict(model, charset, mcfg, path, 'greedy', device=device)[0][0]
    b = predict(model, charset, mcfg, path, 'beam10', device=device)[0][0]
    refs.append(ref); hy_g.append(g); hy_b.append(b)
    ax.imshow(Image.open(path).convert('L'), cmap='gray'); ax.axis('off')
    mark = 'ĐÚNG' if g == ref else 'SAI'
    ax.set_title(f'nhãn: {ref}  |  greedy: {g}  [{mark}]  |  beam10: {b}', fontsize=8, loc='left', pad=1)
plt.tight_layout(); plt.show()
for name, hy in [('greedy', hy_g), ('beam10', hy_b)]:
    m = corpus_metrics(refs, hy)
    print(f'{name:7s} CER {100*m["cer"]:.2f}%  Word Acc {100*m["word_acc"]:.1f}%  (chỉ {len(refs)} ảnh, số liệu rất nhiễu)')

In [ ]:
# 5. Đọc ảnh của bạn (chỉ trên Colab): tải ảnh lên, in n-best kèm xác suất
if IN_COLAB:
    from google.colab import files
    up = files.upload()
    for name in up:
        print(name)
        for i, (t, p) in enumerate(predict(model, charset, mcfg, name, 'beam10', nbest=3, device=device), 1):
            print(f'   #{i}  p={p:.3f}  {t}')
else:
    print('Ô này chỉ tải ảnh lên khi chạy trên Colab. Cục bộ: python demo.py --verbose --decoder beam10 anh.png')

## Tái lập đầy đủ (tùy chọn)

Để chạy toàn bộ pipeline với cấu hình nhỏ (sinh dữ liệu, huấn luyện, đánh giá, phân tích lỗi, ablation, test; khoảng 8-10 phút trên CPU) đặt `RUN_SMOKE = True` rồi chạy ô dưới.
Số liệu của chế độ smoke chỉ để kiểm tra pipeline, không có ý nghĩa về độ chính xác. Huấn luyện đầy đủ (50k ảnh, GPU) xem `README.md`; phần này **chưa được chạy**.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    !bash run_all_smoke.sh
else:
    print('Bỏ qua (RUN_SMOKE = False)')